# Bimanual robot control with LingBot-VLA v2 and OpenVINO

[LingBot-VLA v2](https://github.com/Robbyant/lingbot-vla-v2) is a Vision-Language-Action (VLA) policy from Robbyant.
Given images from the robot cameras, the robot joint state and a task instruction such as
`"Hold the gray kitchenpot with both arms"`, it predicts a chunk of the next 50 robot actions. This notebook
uses the 6B checkpoint fine-tuned on [RoboTwin 2.0](https://robotwin-platform.github.io/), a simulated dual-arm benchmark: each action holds
6 joint positions and 1 gripper position per arm.

The policy has two parts:

* **A Qwen3-VL-4B vision-language backbone.** The three camera views and the instruction run through it once, and the
  key/value cache of all 36 layers forms the context for the action expert.
* **A flow-matching action expert.** A small mixture-of-experts transformer (32 experts, 4 active per token) attends to
  that cache and to the robot state, and turns Gaussian noise into an action chunk in 10 Euler steps.

```
  3 camera views ──┐
                   ├──► Qwen3-VL-4B ──► KV cache ──┐
  instruction ─────┘                               ▼
  robot state, noise ─────────────────────► MoE action expert ──► 50 x 14 actions
                                            (10 flow-matching steps)
```

In this tutorial we load the PyTorch policy from the upstream repository, convert the complete `sample_actions` call,
vision encoder, prefix and the unrolled denoising loop, into a single OpenVINO model, optionally compress its weights
with [NNCF](https://github.com/openvinotoolkit/nncf), and compare the OpenVINO actions with the PyTorch ones. Finally, we
replay a recorded RoboTwin 2.0 episode through the OpenVINO model and render the predicted and the recorded joint
trajectories as a video.

> **Note:** The notebook needs a Linux host with at least 64 GB of RAM (the FP32 PyTorch model is converted on CPU)
> and about 50 GB of free disk space. The upstream code imports Triton, which is only available on Linux.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Download the model](#Download-the-model)
- [Load a RoboTwin 2.0 episode](#Load-a-RoboTwin-2.0-episode)
- [Generate the PyTorch reference and convert the model](#Generate-the-PyTorch-reference-and-convert-the-model)
- [Compress model weights](#Compress-model-weights)
- [Select inference device and model](#Select-inference-device-and-model)
- [Run inference](#Run-inference)
- [Compare with the PyTorch reference](#Compare-with-the-PyTorch-reference)
- [Measure latency](#Measure-latency)
- [Replay the episode](#Replay-the-episode)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/lingbot-vla-v2/lingbot-vla-v2.ipynb" />


## Prerequisites
[back to top ⬆️](#Table-of-contents:)

The model code comes from the upstream repository at a fixed commit, so the versions below follow its requirements:
`transformers` is pinned to 4.57.3, and `lerobot`, which upstream only uses for dataset utilities, is installed without
its dependencies, as the upstream `tools/create_train_env.sh` script does. PyTorch is the CPU build, the policy
runs on CPU only for the reference and the conversion.

In [ ]:
import urllib.request
from pathlib import Path

# Helpers come from `utils` of the cloned repository, otherwise from GitHub; local copies are overwritten
helper_files = ["notebook_utils.py"]
repo_utils = Path("../../utils")
in_repo = (repo_utils / "notebook_utils.py").is_file()
if in_repo:
    helper_contents = {helper_file: (repo_utils / helper_file).read_bytes() for helper_file in helper_files}
else:
    try:
        helper_contents = {}
        for helper_file in helper_files:
            url = f"https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/{helper_file}"
            with urllib.request.urlopen(url, timeout=10) as response:
                helper_contents[helper_file] = response.read()
        if not Path("lingbot_vla_helper.py").exists():
            url = "https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/lingbot-vla-v2/lingbot_vla_helper.py"
            with urllib.request.urlopen(url, timeout=10) as response:
                helper_contents["lingbot_vla_helper.py"] = response.read()
    except OSError as error:
        if any(not Path(helper_file).is_file() for helper_file in helper_files + ["lingbot_vla_helper.py"]):
            raise
        print(f"Could not update helper files, using the existing copies: {error}")
        helper_contents = {}

for helper_file, content in helper_contents.items():
    Path(helper_file).write_bytes(content)

from notebook_utils import setup_notebook_venv

# Requirements installed below go to a notebook-specific virtual environment in the `.venv` folder
setup_notebook_venv()

In [ ]:
%pip install -q "torch==2.8.0" "torchvision==0.23.0" "triton==3.4.0" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q "openvino>=2026.0" "nncf>=3.0"
%pip install -q "transformers==4.57.3" "huggingface-hub>=0.34,<1.0" "accelerate" "safetensors" "einops" "pydantic" "pyyaml"
%pip install -q "torchdata==0.11.0" "datasets==3.6.0" "av" "h5py" "msgpack" "websockets" "pillow" "matplotlib" "pandas" "ipywidgets"
%pip install -q --no-deps "lerobot @ https://github.com/huggingface/lerobot/archive/refs/tags/v0.4.2.tar.gz"

In [ ]:
# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("lingbot-vla-v2.ipynb")

## Download the model
[back to top ⬆️](#Table-of-contents:)

The cell below clones the upstream repository, which has the model code, the robot configs and the action normalization
statistics, and downloads the RoboTwin checkpoint (about 24 GB) from
[Hugging Face](https://huggingface.co/robbyant/lingbot-vla-v2-6b-robotwin). From
[Qwen3-VL-4B-Instruct](https://huggingface.co/Qwen/Qwen3-VL-4B-Instruct) only the configuration and tokenizer files are
needed, the checkpoint already contains the backbone weights.

In [ ]:
from huggingface_hub import snapshot_download

import lingbot_vla_helper as lingbot

REPO_DIR = Path("lingbot-vla-v2-repo")
MODEL_DIR = Path("model")
POLICY_DIR = MODEL_DIR / "lingbot-vla-v2-6b-robotwin"
CKPT_DIR = POLICY_DIR / "checkpoints/global_step_50000/hf_ckpt"
QWEN_DIR = MODEL_DIR / "Qwen3-VL-4B-Instruct"

lingbot.clone_upstream(REPO_DIR)
snapshot_download("robbyant/lingbot-vla-v2-6b-robotwin", local_dir=POLICY_DIR, allow_patterns=["lingbotvla_cli.yaml", "checkpoints/*"])
snapshot_download("Qwen/Qwen3-VL-4B-Instruct", local_dir=QWEN_DIR, allow_patterns=["*.json", "*.txt"])

## Load a RoboTwin 2.0 episode
[back to top ⬆️](#Table-of-contents:)

An observation for the RoboTwin policy holds three RGB images (a head camera and two wrist cameras), the 14 joint
positions of the robot and the task instruction. We take them from an expert demonstration of the `lift_pot` task in the
[RoboTwin 2.0 dataset](https://huggingface.co/datasets/TianxingChen/RoboTwin2.0) (MIT license). Each task is stored as
a zip archive of 50 episodes; `download_robotwin_episode` reads only the recorded data and the instructions of one
episode from it, about 8 MB.

`LingBotPolicy` wraps the upstream `LingbotVLAv2Server`. Upstream targets CUDA, so the wrapper patches a few entry points
(flash-attn, the Triton MoE kernels and `.cuda()` calls) to run on CPU. With `load_weights=False` the model is built on
the `meta` device without weights, enough for the preprocessing: resizing the images to 256x256, tokenizing the
instruction and normalizing the state. Only the image patching moves into the OpenVINO model, so its first input is the
resized images in `[0, 255]`.

In [ ]:
import numpy as np
from PIL import Image

processor = lingbot.LingBotPolicy(REPO_DIR, CKPT_DIR, QWEN_DIR, load_weights=False)
episode = lingbot.RoboTwinEpisode(*lingbot.download_robotwin_episode("data", task="lift_pot", episode=0))
observation = episode.observation(0)
inputs = processor.preprocess(observation)

print(f"episode: {len(episode)} frames")
print(f"instruction: {observation['task']}")
print(f"joint state: {np.round(observation['observation.state'], 3)}")
for name, value in inputs.items():
    print(f"{name:15s} {str(value.shape):22s} {value.dtype}")

display(Image.fromarray(np.hstack([observation[camera] for camera in lingbot.CAMERAS])))

## Generate the PyTorch reference and convert the model
[back to top ⬆️](#Table-of-contents:)

Next we load the PyTorch weights, run `sample_actions` with a fixed noise sample to get reference actions, and convert
the policy with `ov.convert_model`. A few parts of the upstream forward are data-dependent or written for CUDA, so
`convert_to_openvino` swaps them for traceable versions with the same math before tracing:

* **MoE experts.** Upstream dispatches tokens to experts with Triton grouped-GEMM kernels. For the conversion every expert
  runs on every token as two wide matrix multiplications, and the routing weights (zero for the experts that were not
  selected) scale the activations before the down projection, which also sums the expert outputs.
* **Attention.** The ViT attention of the three images runs as one batched `scaled_dot_product_attention` instead of a
  per-image loop, and the action expert uses `scaled_dot_product_attention` instead of its eager implementation.
* **Static index computations.** The image grid is fixed, so the vision rotary and position embeddings become constants,
  and the multimodal RoPE position ids are computed with tensor operations instead of a Python loop.
* **Denoising loop.** The 10 Euler steps are unrolled, so one OpenVINO model maps images, instruction tokens, state and
  noise to actions.

Before tracing, the wrapper output is checked against the reference, and the model is saved with FP16 weights (about
12 GB). Loading and converting takes a few minutes and up to about 55 GB of RAM; the step is skipped when the model
already exists.

In [ ]:
import gc

OV_DIR = MODEL_DIR / "ov"
FP16_XML = OV_DIR / "fp16/lingbot_vla_v2.xml"
REFERENCE = OV_DIR / "reference.npz"

if FP16_XML.exists() and REFERENCE.exists():
    print(f"{FP16_XML} already exists, skipping conversion")
else:
    pt_policy = lingbot.LingBotPolicy(REPO_DIR, CKPT_DIR, QWEN_DIR)
    noise = np.random.default_rng(0).standard_normal((1, pt_policy.config.n_action_steps, pt_policy.config.max_action_dim)).astype(np.float32)
    reference_actions = pt_policy.sample_actions(inputs, noise)
    lingbot.convert_to_openvino(pt_policy, inputs, noise, reference_actions, FP16_XML)
    np.savez(REFERENCE, noise=noise, actions=reference_actions)
    del pt_policy
    gc.collect()

reference = np.load(REFERENCE)
print(f"reference actions: {reference['actions'].shape}")

## Compress model weights
[back to top ⬆️](#Table-of-contents:)

Inference of this model is bound by memory bandwidth: every denoising step reads the weights of the action expert again.
[NNCF](https://github.com/openvinotoolkit/nncf) weight compression shrinks them with little effect on the actions.
`compress_mixed` applies two precisions:

* INT4 (asymmetric, group size 128) to the routed MoE experts of the action expert, which hold most of the weights read
  per denoising step;
* INT8 (asymmetric, per channel) to the remaining matrix multiplications of the backbone and the action expert.

Small layers such as the MoE routers and the state and action projections stay in FP16. Compression takes a few minutes
and about 20 GB of RAM. Uncheck the box to skip it.

In [ ]:
from notebook_utils import quantization_widget

to_compress = quantization_widget()

to_compress

In [ ]:
import openvino as ov

COMPRESSED_XML = OV_DIR / "int4_int8/lingbot_vla_v2.xml"

if to_compress.value and not COMPRESSED_XML.exists():
    core = ov.Core()
    compressed = lingbot.compress_mixed(core.read_model(FP16_XML))
    ov.save_model(compressed, COMPRESSED_XML)
    del compressed
    gc.collect()

for xml in (FP16_XML, COMPRESSED_XML):
    if xml.exists():
        print(f"{str(xml):45s} {xml.with_suffix('.bin').stat().st_size / 2**30:5.1f} GB")

## Select inference device and model
[back to top ⬆️](#Table-of-contents:)

Select the device and the model variant. An Intel GPU is recommended: on CPU one call takes several seconds. The FP16
model needs about 13 GB of GPU memory, the compressed one about 6 GB.

In [ ]:
from notebook_utils import device_widget

core = ov.Core()
gpu = next((d for d in core.available_devices if d.startswith("GPU")), "CPU")
device = device_widget(default=gpu, exclude=["NPU", "AUTO"])

device

In [ ]:
import ipywidgets as widgets

variants = {"FP16": FP16_XML, "INT4-INT8": COMPRESSED_XML}
model_variant = widgets.Dropdown(
    options=[name for name, xml in variants.items() if xml.exists()],
    value="INT4-INT8" if COMPRESSED_XML.exists() else "FP16",
    description="Model:",
)

model_variant

## Run inference
[back to top ⬆️](#Table-of-contents:)

`OVLingBotPolicy` compiles the model with the `LATENCY` performance hint; the first compilation on GPU takes a couple of
minutes. We run it with the same noise as the PyTorch reference. The output is the normalized action chunk with
padding up to 55 dimensions; `postprocess` keeps the 14 RoboTwin dimensions and undoes the normalization with the
training statistics.

In [ ]:
ov_policy = lingbot.OVLingBotPolicy(variants[model_variant.value], device.value, core)
print(f"compiled {model_variant.value} model for {device.value} in {ov_policy.compile_time:.0f} s")

ov_actions = ov_policy(inputs, reference["noise"])
actions = processor.postprocess(observation, ov_actions)["action"]
print(f"normalized output: {ov_actions.shape}, robot actions: {actions.shape}")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

columns = [f"left_joint{i}" for i in range(6)] + ["left_gripper"] + [f"right_joint{i}" for i in range(6)] + ["right_gripper"]
display(pd.DataFrame(np.round(actions[:5], 4), columns=columns, index=[f"t+{i}" for i in range(5)]))

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, arm, offset in zip(axes, ("left", "right"), (0, 7)):
    for i in range(7):
        ax.plot(actions[:, offset + i], label=columns[offset + i].split("_", 1)[1])
    ax.set_title(f"{arm} arm")
    ax.set_xlabel("step")
axes[0].set_ylabel("position")
axes[1].legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()

## Compare with the PyTorch reference
[back to top ⬆️](#Table-of-contents:)

With the same inputs and noise, the OpenVINO actions should match the PyTorch FP32 reference up to the precision of
the weights and of the inference. We compare the normalized model outputs with cosine similarity and the largest
absolute difference. On an Intel Arc B390 GPU the FP16 model reaches a cosine similarity of about 0.99998 and the
compressed model about 0.9999.

In [ ]:
cosine = lingbot.cos_sim(ov_actions, reference["actions"])
max_diff = np.abs(ov_actions - reference["actions"]).max()
print(f"{model_variant.value} on {device.value}: cosine similarity {cosine:.6f}, max |diff| {max_diff:.4f}")

## Measure latency
[back to top ⬆️](#Table-of-contents:)

Finally, we measure the median latency of one `sample_actions` call: the vision encoder, the backbone prefix and the 10
denoising steps. Preprocessing and postprocessing run in Python and are not included. On an Intel Arc B390 GPU we
measured about 480 ms for the FP16 model and 255-280 ms for the compressed one.

In [ ]:
latency = ov_policy.benchmark(inputs, reference["noise"], warmup=2, iterations=10)
print(f"{model_variant.value} on {device.value}: {latency:.0f} ms per action chunk ({1000 / latency:.1f} chunks/s)")

## Replay the episode
[back to top ⬆️](#Table-of-contents:)

To see how the policy behaves on real observations, we replay the recorded episode open-loop: every 10 frames the
policy gets the recorded camera images and joint state and predicts the next 50 actions, and we compare them with the
joint positions the expert actually reached. The policy does not control the robot here, which would require the
RoboTwin simulator, so errors do not accumulate.

The video shows the three camera views and, for a few joints, the recorded trajectory (gray) and the latest predicted
action chunk (orange).

In [ ]:
from IPython.display import Video

rng = np.random.default_rng(0)
chunks = {}
for t in range(0, len(episode), 10):
    frame = episode.observation(t)
    chunks[t] = processor.postprocess(frame, ov_policy(processor.preprocess(frame), rng.standard_normal(ov_policy.noise_shape, np.float32)))["action"]
    recorded = episode.joints[t + 1 : t + 1 + len(chunks[t])]
    print(f"frame {t:3d}: mean |predicted - recorded| {np.abs(chunks[t][: len(recorded)] - recorded).mean():.4f}")

video_path = lingbot.render_replay_video(episode, chunks, Path("data/lingbot_vla_v2_replay.mp4"))
Video(video_path, embed=True, width=960)